In [1]:
import numpy as np
import pandas as pd

import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

# for visualization
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

import warnings
warnings.filterwarnings('ignore')


In [2]:
# loading diabetis dataset from github

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']


df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
print(df.info())
print("---------------------------------------------------------------")
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None
---------------------------------------------------------------


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [4]:
# in the above dataset all the missing values are represented by 0, hence it will not show missing values, hence we have to change 0 to 'NaN'

df.isna().sum()     

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [5]:

# Replacing zero values with NaN in columns 
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Imputing missing values with mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [6]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [ ]:
# making a function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)       # we are asking to suggest the best estimator starting from 50 to 200, based on the past data
    max_depth = trial.suggest_int('max_depth', 3, 20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score


In [19]:

# Creating a study object and optimizing the objective function
study = optuna.create_study(
    direction = 'maximize', 
    sampler = optuna.samplers.TPESampler(),
    pruner = optuna.pruners.HyperbandPruner()
)  # aiming to maximize accuracy

study.optimize(objective, n_trials=50)  # Running 50 trials to find the best hyperparameters

[I 2026-03-09 21:57:57,960] A new study created in memory with name: no-name-154e29e7-e418-4c4a-95b3-7c999e6ad1ca
[I 2026-03-09 21:57:58,881] Trial 0 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 94, 'max_depth': 8}. Best is trial 0 with value: 0.7616387337057727.
[I 2026-03-09 21:57:59,616] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 178, 'max_depth': 18}. Best is trial 1 with value: 0.7672253258845437.
[I 2026-03-09 21:57:59,941] Trial 2 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 83, 'max_depth': 20}. Best is trial 1 with value: 0.7672253258845437.
[I 2026-03-09 21:58:00,370] Trial 3 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 98, 'max_depth': 8}. Best is trial 1 with value: 0.7672253258845437.
[I 2026-03-09 21:58:00,706] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 86, 'max_depth': 7}. Best is trial 4 with value: 0.76908752327

### Optuna: Deep Dive into `create_study()`

The `optuna.create_study()` function is the starting point of any Optuna optimization process. It initializes a "Study" object, which is responsible for managing the entire history of trials, tracking the best hyperparameters, and deciding where the algorithm should search next.

Understanding the parameters inside `create_study()` allows you to control exactly how the optimization behaves, from standardizing randomness to saving progress halfway through.

Here is an in-depth look at its parameters.

### 1. `direction` (String)
*   **What it does:** Tells Optuna what your ultimate goal is regarding the evaluation metric you return in your objective function.
*   **Options:**
    *   `'maximize'`: Use this if you are trying to get the highest score possible (for accuracy) (e.g., Accuracy, F1-Score, R-squared).
    *   `'minimize'`: Use this if you are trying to get the lowest score possible (for errors) (e.g., Mean Squared Error, Log Loss).
*   **Default:** `'minimize'`

### 2. `directions` (List of Strings)
*   **What it does:** Used strictly for Multi-Objective Optimization. If your objective function returns *two or more* scores (for example, you want high accuracy BUT ALSO fast prediction time), you pass a list of directions.
*   **Example:** `directions=['maximize', 'minimize']`
*   **Note:** You cannot use `direction` and `directions` at the same time.

### 3. `study_name` (String)
*   **What it does:** Gives your study a custom, readable name instead of a random UUID.
*   **Why it's useful:** If you are saving your studies to a database (using the `storage` parameter), giving it a name is required so you can easily find it and resume it later.

### 4. `storage` (String or Optuna Storage Object)
*   **What it does:** By default, Optuna saves the history of all trials in the computer's RAM. If the script crashes, you lose everything. `storage` allows you to save the progress to a permanent database (like SQLite, PostgreSQL, or MySQL).
*   **Example:** `storage='sqlite:///my_study.db'`
*   **Why it's useful:** It allows you to pause a study and resume it the next day. It also allows multiple computers (distributed computing) to work on the exact same study simultaneously by pointing them to the same database URL.

### 5. `sampler` (Optuna Sampler Object)
*   **What it does:** Defines the exact algorithm Optuna uses to guess the next set of hyperparameters. 
*   **Options:**
    *   `TPESampler()`: The default. Uses the Tree-structured Parzen Estimator (Bayesian Optimization). This is almost always the best choice.
    *   `RandomSampler()`: Blindly picks random combinations (equivalent to RandomSearchCV).
    *   `GridSampler()`: Exhaustively tries combinations from a predefined grid (equivalent to GridSearchCV).
    *   `CmaEsSampler()`: Excellent for continuous mathematical functions but overkill for standard ML algorithms.
*   **Why it's useful:** You can pass arguments into the sampler, such as setting a random seed to make your optimization 100% reproducible: `sampler=optuna.samplers.TPESampler(seed=42)`.

### 6. `pruner` (Optuna Pruner Object)
*   **What it does:** Defines the algorithm used to kill bad, unpromising trials early to save time. 
*   **Options:**
    *   `MedianPruner()`: The default. It kills a trial if its intermediate score is worse than the median score of all previous trials at the same step.
    *   `HyperbandPruner()`: A more aggressive, advanced pruner that allocates resources to the best-performing models very quickly.
*   **Why it's useful:** If you are training a Deep Learning model that takes 50 epochs, the pruner will stop the training at epoch 5 if it predicts the final result will be terrible.

### 7. `load_if_exists` (Boolean)
*   **What it does:** Used in conjunction with `study_name` and `storage`. 
*   **Options:**
    *   `False` (Default): If you try to create a study with a name that already exists in your database, Optuna will throw a `DuplicatedStudyError`.
    *   `True`: Instead of crashing, Optuna will simply load the massive history of the past study and seamlessly continue adding new trials to it.

---

### Example: A fully configured `create_study`

```python
import optuna

study = optuna.create_study(
    direction='maximize',                     # We want high accuracy
    study_name='xgboost_optimization_v1',     # Give it a readable name
    storage='sqlite:///optuna_history.db',    # Save it to a local SQL file so we don't lose progress
    load_if_exists=True,                      # If we run this script again tomorrow, just pick up where we left off
    sampler=optuna.samplers.TPESampler(seed=42) # Make the randomness reproducible
)

# Start the optimization
# study.optimize(objective, n_trials=100)


In [26]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')
study.best_trial

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 116, 'max_depth': 19}


FrozenTrial(number=16, state=<TrialState.COMPLETE: 1>, values=[0.7802607076350093], datetime_start=datetime.datetime(2026, 3, 9, 21, 58, 6, 537166), datetime_complete=datetime.datetime(2026, 3, 9, 21, 58, 7, 315792), params={'n_estimators': 116, 'max_depth': 19}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=200, log=False, low=50, step=1), 'max_depth': IntDistribution(high=20, log=False, low=3, step=1)}, trial_id=16, value=None)

In [29]:
# Training a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74
